# pyjmri exploration notebook

This notebook walks through the four things you most often want to do interactively against a running JMRI server:

1. Open a `Client` and discover the layout.
2. Change a specific turnout by system name.
3. Acquire several locomotives at once.
4. Send commands to each locomotive independently, in any order, from any cell.

Run the cells top-to-bottom once. After that the `jmri`, `layout`, and throttle handles (`t1`, `t2`, `t3`) stay bound for the rest of the session, so you can re-run any command cell as often as you like.

**Target**: `192.168.1.159:12080` is the layout machine in the basement. Commands sent here actually move trains and throw turnouts.

## 1. Imports and logging

`TurnoutState` is the enum used for `CLOSED` / `THROWN`. The logging config writes WebSocket traffic to `pyjmri.log` next to the notebook; switch to `level=logging.INFO` if the DEBUG output is too noisy.

In [ ]:
import logging
from pyjmri import Client, TurnoutState  # pyright: ignore[reportMissingImports]

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",
    force=True,
)

## 2. Open the client and discover the layout

`Client.__aenter__` opens the HTTP and WebSocket connections and keeps them open for the rest of the notebook. `jmri.discover()` returns a `Layout` object whose collections (`turnouts`, `sensors`, `blocks`, ...) are populated from JMRI's current state.

Re-running this cell on an already-open client raises `RuntimeError` — run the **shutdown** cell at the bottom first if you need to reconnect.

In [3]:
jmri = await Client("192.168.1.159:12080").__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors, {len(layout.blocks)} blocks")

discovered: 52 turnouts, 63 sensors, 38 blocks


## 3. Change a specific turnout

Look the turnout up by system name (`NT102`, `NT100`, ...) or by user name if one is assigned. `set_state` sends the command to JMRI; remember that NCE is open-loop, so the state JMRI reports is *the last commanded state*, not an observed one.

In [14]:
turnout = layout.turnouts.by_system_name("NT102")
print(f"name={turnout.name}  user_name={turnout.user_name}  current state={turnout.state.name}")

name=NT102  user_name=South Turnout 102  current state=THROWN


In [9]:
await layout.turnouts.by_system_name("NT102").set_state(TurnoutState.CLOSED)

In [12]:
await layout.turnouts.by_system_name("NT102").set_state(TurnoutState.THROWN)

## 4. Acquire several locomotives

`layout.throttle(dcc_address, long=...)` returns a `Throttle` that is an async context manager. The `async with` form is convenient for one-shot use inside a single cell, but it releases the throttle as soon as the block ends — no good if you want to drive the loco from later cells.

To keep a throttle alive across cells, enter the context manually with `__aenter__()` and bind it to a variable. Each throttle has its own WebSocket correlation and heartbeat task, so the three handles below are fully independent.

Use `long=True` for DCC addresses ≥ 128, `long=False` for short addresses.

In [18]:
t1 = await layout.throttle(2570, long=True).__aenter__()
t2 = await layout.throttle(8096, long=True).__aenter__()
t3 = await layout.throttle(5488,   long=True).__aenter__()
print("acquired three throttles")

acquired three throttles


## 5. Drive each locomotive independently

Every cell below acts on exactly one throttle. Run them in any order, re-run them as often as you like — the throttle handles persist until you release them.

Speed is a float from `0.0` (stop) to `1.0` (full). `forward=False` reverses direction.

In [ ]:
await t1.set_speed(0.4, forward=True)

In [ ]:
await t2.set_speed(0.3, forward=True)

In [26]:
await t3.set_speed(0.1, forward=False)

In [ ]:
await t1.set_speed(0.0, forward=True)

In [ ]:
await t2.set_speed(0.0, forward=False)

In [27]:
await t3.set_speed(0.0, forward=True)

In [ ]:
t3

## 6. Run multiple locomotives concurrently

There is only one event loop in the notebook, but you can run as many *tasks* on it as you like. Two patterns cover almost everything:

- **`asyncio.gather(...)`** — start several coroutines and wait for them all to finish. The cell unblocks when the slowest one is done.
- **`asyncio.create_task(...)`** — start a coroutine in the background and return immediately, so you can keep working in other cells. Join later with `await task`, or abort with `task.cancel()`.

Plain sequential `await run_for(t1, ...)` followed by `await run_for(t2, ...)` does **not** overlap — the second call only starts once the first finishes. Use `gather` or `create_task` whenever you want true concurrency.

The helper below puts a loco at a target speed, sleeps, then forces it back to 0 — even if the task is cancelled mid-run.

In [19]:
import asyncio

async def run_for(throttle, speed, seconds, forward=True):
    try:
        await throttle.set_speed(speed, forward=forward)
        await asyncio.sleep(seconds)
    finally:
        await throttle.set_speed(0.0, forward=forward)

### Pattern A — `asyncio.gather`: run t1 for 10 s and t2 for 5 s in parallel

Both locos start at the same instant; t2 stops after 5 s, t1 keeps going until 10 s, and the cell returns once the longer run finishes.

In [20]:
await asyncio.gather(
    run_for(t1, 0.2, 10),
    run_for(t2, 0.2, 10),
    run_for(t3, 0.2, 10)
)

[None, None, None]

### Pattern B — `asyncio.create_task`: start the runs in the background

The cell returns immediately, so you can keep issuing commands (or watch sensor traffic) while the trains move. Later cells can `await task1` to join, or `task1.cancel()` to abort cleanly — the `finally` in `run_for` guarantees the throttle is set back to 0.

In [23]:
task1 = asyncio.create_task(run_for(t1, 0.1, 10))
task2 = asyncio.create_task(run_for(t2, 0.1, 5))
task3 = asyncio.create_task(run_for(t3, 0.1, 10))
print("both tasks running in background")

both tasks running in background


In [24]:
await asyncio.gather(task1, task2)
print("both tasks finished")

both tasks finished


In [25]:
await asyncio.gather(task3)

[None]

In [ ]:
for task in (task1, task2):
    if not task.done():
        task.cancel()
print("any running tasks cancelled")

## 7. Why does `turnout.state` look stale after `set_state`?

A common confusion: you call `set_state(CLOSED)` on NT102, re-run the inspect cell, and the printed state is the same value it had before. That is **expected** — and the same caching rule applies to every entity in the layout (sensors, blocks, lights, memories).

The cached `.state` attribute is updated in only three ways:

1. **Initial `discover()`** captures the state at startup.
2. **`await entity.get_state()`** forces a one-shot HTTP refresh.
3. **A WebSocket push event** arrives — but only after the entity has been *subscribed*. Subscription is established lazily, the first time you call `wait_state(...)`, `wait_change(...)`, or `set_state(..., wait_for_jmri_state=True)` on that entity.

`set_state(state)` on its own sends the command and returns once JMRI ack'd it over HTTP. It does **not** touch the local cache. So a bare `print(turnout.state.name)` keeps showing the discovery-time value until something refreshes it.

Two fixes, depending on what you want:

- **Quick refresh** — `await turnout.get_state()` before printing.
- **Confirmed command + cache update in one call** — `await turnout.set_state(TurnoutState.CLOSED, wait_for_jmri_state=True)`. This subscribes, sends the command, waits for JMRI's WS echo, and updates the cache as a side effect. From that point on, any subsequent state change JMRI broadcasts will also keep the cache live.

Honesty caveat (FR22): even `wait_for_jmri_state=True` only confirms JMRI's *commanded* state, not the physical position of the points. NCE is open-loop — no hardware feedback. The cache reflects what JMRI was told, not what the layout actually did.

In [ ]:
turnout = layout.turnouts.by_system_name("NT102")
print(f"cached before refresh: {turnout.state.name}")
await turnout.get_state()
print(f"cached after refresh:  {turnout.state.name}")

## 8. Sensors

Sensors are read-only in pyjmri v1 — they report layout state (block detectors, push-buttons, etc.) but you don't drive them from the API. The cache rules from §7 apply: the snapshot right after `discover()` is the state JMRI last reported; for fresh values either call `await sensor.get_state()` or use a `wait_*` primitive (which auto-subscribes and keeps the cache live from then on).

In [ ]:
for s in layout.sensors.values():
    print(f"{s.name:8} {s.user_name or '':40} {s.state.name}")

In [ ]:
# Refresh one sensor explicitly. Pick any sensor from the snapshot above.
sensor = next(iter(layout.sensors.values()))
state = await sensor.get_state()
print(f"{sensor.name} ({sensor.user_name or '-'}) -> {state.name}")

In [ ]:
# Wait for a specific sensor to go ACTIVE (will block until it does, or
# timeout fires). Pick a real sensor name from the snapshot above.
#
# from pyjmri import SensorState
# s = layout.sensors.by_system_name("NS401")
# new_state = await s.wait_state(SensorState.ACTIVE, timeout=30.0)
# print(f"{s.name} is now {new_state.name}")

### Non-blocking sensor waits — "drive until something happens"

`await sensor.wait_active()` looks blocking, but the only thing it suspends is the *current coroutine*. The asyncio event loop is free, other tasks keep running, and — crucially — the locomotive keeps moving while you wait. Throttle commands set state on NCE; the loco doesn't need a continuous stream of speed packets to keep rolling.

That means the meaningful-script pattern is just an `async def`:

```python
await throttle.set_speed(0.3, forward=True)
await sensor.wait_active()        # suspended here, train still moving
await throttle.set_speed(0.0, forward=True)
```

Two practical wrappers below:

1. **`drive_until_active`** — set speed, await a sensor, stop. A `try/finally` guarantees the loco is brought back to zero even if the caller cancels the task or a timeout fires.
2. **`shuttle_between`** — run forward to one sensor, reverse to another, repeat. Exactly the "do X until sensor A, then Y until sensor B" pattern you described.

Either function can be awaited inline (the cell sits there until the run finishes) **or** wrapped in `asyncio.create_task(...)` and run in the background while other cells stay interactive — the same pattern §6 uses for concurrent throttle runs. Cancelling the task triggers the `finally` clause and stops the train cleanly.

One gotcha: `wait_active` returns immediately if the cached state is already `ACTIVE`. If you want "wait for the *next* activation regardless of current state," use `await sensor.wait_change()` and filter the returned state, or read `sensor.state` first and pick the appropriate `wait_*` call.

In [ ]:
async def drive_until_active(throttle, sensor, speed, *, forward=True, timeout=None):
    """Move a loco at ``speed`` until ``sensor`` reports ACTIVE, then stop.

    Cancelling the awaiting task (or hitting the timeout) still runs the
    finally clause, so the loco is always brought back to zero.
    """
    try:
        await throttle.set_speed(speed, forward=forward)
        await sensor.wait_active(timeout=timeout)
    finally:
        await throttle.set_speed(0.0, forward=forward)

In [ ]:
async def shuttle_between(throttle, sensor_a, sensor_b, speed=0.3, laps=1):
    """Run forward until sensor_a fires, reverse until sensor_b fires, repeat."""
    try:
        for _ in range(laps):
            await throttle.set_speed(speed, forward=True)
            await sensor_a.wait_active()
            await throttle.set_speed(speed, forward=False)
            await sensor_b.wait_active()
    finally:
        await throttle.set_speed(0.0, forward=True)

In [ ]:
# Background pattern — kick the run off so the notebook stays responsive.
#
# sensor_a = layout.sensors.by_system_name("NS401")
# sensor_b = layout.sensors.by_system_name("NS402")
# shuttle = asyncio.create_task(shuttle_between(t1, sensor_a, sensor_b, speed=0.3))
#
# # Cell returns immediately — keep working in other cells. To stop early:
# # shuttle.cancel()
# # await asyncio.gather(shuttle, return_exceptions=True)
#
# # To join when the shuttle finishes naturally:
# # await shuttle

## 9. Block occupancy — snapshot + live monitor

Blocks model occupancy regions on the layout and are read-only in v1. The most useful pattern is **watching every block at once** while a train moves, printing each change as it happens.

The watcher below spawns one background task per block, each looping on `wait_change()`. Each call auto-subscribes that block to JMRI's WebSocket stream, so when the train rolls into or out of a detected section the matching task prints a line and loops back to wait for the next transition.

`UNDETECTED` is a real and distinct value from `UNKNOWN` — it means the block has no occupancy detector wired, not that JMRI hasn't reported anything yet. pyjmri preserves both faithfully (FR14), so you can spot un-instrumented territory in the snapshot below.

Workflow: run the snapshot to see the starting picture, start the monitor, drive a loco from the throttle cells in §5/§6, watch the print stream, then cancel the monitor when you're done.

In [ ]:
for b in layout.blocks.values():
    label = b.user_name or ""
    print(f"{b.name:8} {label:35} {b.state.name:11} value={b.value!r}")

In [ ]:
import asyncio
import time

async def _watch_block(block):
    while True:
        new_state = await block.wait_change()
        stamp = time.strftime("%H:%M:%S")
        label = block.user_name or block.name
        print(f"[{stamp}] {block.name:8} ({label}) -> {new_state.name}")

block_monitors = [asyncio.create_task(_watch_block(b)) for b in layout.blocks.values()]
print(f"watching {len(block_monitors)} blocks — run a train, then cancel below")

In [ ]:
for task in block_monitors:
    task.cancel()
# Drain the cancellations so they don't show up as pending warnings.
await asyncio.gather(*block_monitors, return_exceptions=True)
print("block monitor stopped")

## 10. Activating routes

A JMRI route is a saved sequence of turnout positions — firing it throws every turnout in the route to its saved position in one call. Routes in pyjmri are *trigger-only*: there is no observable "active" persistent state to read back (JMRI reports `state=0` after activation), so `Route` exposes only `name`, `user_name`, and `activate()`. As with every other command in this library, NCE is open-loop — the route activation is only a *commanded* outcome, not a confirmed physical one.

In [ ]:
for r in layout.routes.values():
    print(f"{r.name:10} {r.user_name or ''}")

In [ ]:
# Activate one route by user or system name. Replace with a real name
# from the list above.
# await layout.routes["My Route"].activate()

## 11. Memory variables

JMRI "memory" objects are named string slots used by panels, LogixNG, and scripts to share values (train IDs, dispatcher notes, signal hints, etc.). They are not part of physical layout control, but they are a useful integration point.

`set_value` does **not** update the local cache — call `get_value()` afterward if you want to see the new value reflected on the Python side. Memory has no WebSocket push plumbing in v1, so there is no `wait_*` API for memories.

In [ ]:
for m in layout.memories.values():
    print(f"{m.name:14} {m.user_name or '':30} value={m.value!r}")

In [ ]:
# Replace IM... with a real memory system name from the list above.
# mem = layout.memories.by_system_name("IMCURRENTTIME")
# await mem.set_value("3001")
# print("readback:", await mem.get_value())

## 12. Shutdown

Stop every loco, release each throttle, then close the client. Releasing throttles before closing the client avoids leaving orphaned throttle sessions on the JMRI side. Restarting the kernel also cleans everything up if you forget.

If you started the block monitor in §9 and haven't cancelled it, do that first — otherwise the monitor tasks will keep firing into a closing WebSocket.

In [28]:
for t in (t1, t2, t3):
    try:
        await t.set_speed(0.0, forward=True)
        await t.release()
    except Exception as e:
        print(f"release failed (probably already released): {e}")
print("throttles released")

throttles released


In [31]:
await jmri.__aexit__(None, None, None)
print("client closed")

client closed


## 13. Moving from notebook to standalone script

This notebook works because Jupyter has already started an asyncio event loop for you — that's why bare `await client.discover()` at the top of a cell just works. A regular `.py` file has no loop running yet, so there's a one-line change at the entry point, plus a recommended switch to `async with` for the context managers. Before any of that, though, you want a clean project of your own — not the `python_code/` repo, which exists to build and ship pyjmri itself.

### Start a fresh project from a virgin terminal

The `pyproject.toml` in this repository describes the **pyjmri library** — its own dependencies, build system, version, etc. Adding your script's dependencies to it would mix application code into the library's manifest. Once pyjmri is on PyPI you should treat it like any other third-party package: make a new directory, initialise it with `uv`, add `pyjmri` as a dependency, and work from there.

```
mkdir ~/my-jmri-scripts
cd ~/my-jmri-scripts
uv init                              # creates pyproject.toml, .python-version, hello.py, .gitignore
uv add pyjmri                        # pulls the published wheel from PyPI
uv add httpx                         # or whatever else your script needs
```

After `uv init` you have your own `pyproject.toml`, a `.venv/` (created lazily on first `uv run`), and a `uv.lock` that pins your environment. `uv add` updates both `pyproject.toml` and the lockfile.

Layout of a typical script project:

```
my-jmri-scripts/
  pyproject.toml          # YOUR manifest — pyjmri listed under [project.dependencies]
  uv.lock                 # YOUR lockfile
  .venv/                  # YOUR virtualenv
  shuttle.py              # entry-point script
  drive_helpers.py        # shared helpers, imported by shuttle.py
```

Run a script with:

```
uv run python shuttle.py
uv run python shuttle.py --url 192.168.1.159:12080
```

`uv run` resolves the lockfile, makes sure `.venv` matches, then invokes Python from inside it. You do not need to `source .venv/bin/activate` first. If you've just changed `pyproject.toml`, the next `uv run` re-syncs automatically.

### When to use the pyjmri repo's pyproject.toml

Only when you're hacking on pyjmri itself — fixing a bug in the library, adding a feature, running its test suite. In that case scripts under `python_code/examples/` are part of the dev tree, and `uv run --no-sync python examples/...` from inside `python_code/` is the right invocation (see `examples/hello_jmri.py`). For everything else — your own automation, your own layout scripts — make a new project as above. Treat the `python_code/` repo as if it were `numpy`'s source tree: you don't write your application inside it.

### Importing sibling modules

Python adds the *script's own directory* to `sys.path` automatically, so co-located helpers just work:

```python
# shuttle.py
from drive_helpers import drive_until_active, shuttle_between
```

If you grow past a handful of files, promote `drive_helpers` to a real package — make it a directory with an `__init__.py`, or declare it in `pyproject.toml` under `[tool.hatch.build.targets.wheel]` (or `[tool.setuptools.packages]`, depending on your build backend) so it's importable no matter where you run from.

### Event loop — the actual difference

Jupyter runs `ipykernel`'s event loop in the background. Every code cell is wrapped so `await` at the top level is legal. In a `.py` file:

- There is no loop until you start one.
- `await` is a syntax error outside an `async def`.
- You need **one** entry point that calls `asyncio.run(main())`. That call creates the loop, runs your coroutine to completion, then closes the loop. Call it exactly once, and never from inside another running loop.

Minimal skeleton:

```python
import asyncio
from pyjmri import Client, TurnoutState

async def main() -> None:
    async with Client("192.168.1.159:12080") as jmri:
        layout = await jmri.discover()
        await layout.turnouts.by_system_name("NT102").set_state(
            TurnoutState.CLOSED, wait_for_jmri_state=True
        )

if __name__ == "__main__":
    asyncio.run(main())
```

### Notebook idiom vs script idiom — side by side

| Concern | Notebook (this file) | Standalone script |
| --- | --- | --- |
| Start the loop | Already running | `asyncio.run(main())` once, at the bottom |
| Open the client | `jmri = await Client(...).__aenter__()` (manual, so handles survive across cells) | `async with Client(...) as jmri:` (automatic teardown) |
| Acquire a throttle | `t1 = await layout.throttle(2570, long=True).__aenter__()` | `async with layout.throttle(2570, long=True) as t1:` |
| Shut down | Explicit `release()` + `__aexit__` cells | Falls out of `async with`; nothing to remember |
| Top-level `await` | Legal in every cell | Illegal — wrap in `async def` |
| Background tasks | `asyncio.create_task(...)` returns to the cell; you can poke at the task from later cells | `asyncio.create_task(...)` only inside `main()`; if `main()` returns, the task is cancelled with the loop |

The notebook uses manual `__aenter__()` *only* because handles need to survive across cells. In a script always prefer `async with` — it guarantees the WebSocket and HTTP session are closed even if your code raises, and it stops you accidentally leaking throttle sessions on the JMRI side.

### Putting it together — a runnable shuttle script

```python
# ~/my-jmri-scripts/shuttle.py
import asyncio
from pyjmri import Client

async def shuttle_between(throttle, sensor_a, sensor_b, speed=0.3, laps=1):
    try:
        for _ in range(laps):
            await throttle.set_speed(speed, forward=True)
            await sensor_a.wait_active()
            await throttle.set_speed(speed, forward=False)
            await sensor_b.wait_active()
    finally:
        await throttle.set_speed(0.0, forward=True)

async def main() -> None:
    async with Client("192.168.1.159:12080") as jmri:
        layout = await jmri.discover()
        sensor_a = layout.sensors.by_system_name("NS401")
        sensor_b = layout.sensors.by_system_name("NS402")
        async with layout.throttle(2570, long=True) as t:
            await shuttle_between(t, sensor_a, sensor_b, speed=0.3, laps=3)

if __name__ == "__main__":
    asyncio.run(main())
```

Run it from your project directory:

```
cd ~/my-jmri-scripts
uv run python shuttle.py
```

### Things that bite

- **`RuntimeError: asyncio.run() cannot be called from a running event loop`** — you're using the script idiom inside Jupyter, or calling `asyncio.run` from a coroutine. In a notebook just `await` directly; from inside a coroutine call the inner coroutine with `await`, not `asyncio.run`.
- **Background tasks vanishing at exit** — every `create_task` lives only as long as the loop. In a script, make sure `main()` awaits or joins any background work before it returns, or wrap the body in `try/finally` and cancel + `gather(..., return_exceptions=True)` your tasks on the way out (same pattern §9 uses to stop the block monitor).
- **`KeyboardInterrupt` mid-run** — `asyncio.run` re-raises it after closing the loop. If your loco needs to stop on Ctrl-C, catch `KeyboardInterrupt` (or `asyncio.CancelledError` inside `main()`) and zero the throttle in a `finally`, the same shape as `drive_until_active` in §8.
- **`uv add` editing the wrong `pyproject.toml`** — `uv` walks up from the current directory looking for the nearest `pyproject.toml`. If you accidentally run `uv add pyjmri` from anywhere inside `python_code/`, it will try to add pyjmri as a dependency of pyjmri. Always `cd` into your own project first.
- **Logging** — `logging.basicConfig(...)` configures the root logger once. In a notebook re-running the cell silently no-ops unless you pass `force=True` (which this notebook does). In a script the first call wins; place it at the top of `main()` or in an `if __name__ == "__main__":` block.